# Phase 3 Real Stimulus-Based Bayesian Multilevel Model

This notebook fits the final pre-specified empirical stimulus model on the cleaned real participant dataset.

Primary model:

`rating ~ episode + group + (1 | participant_id) + (1 | stimulus_id)`

The model uses a Gaussian likelihood in native 0-100 rating units, participant and stimulus random intercepts, and fixed effects for episode and group. The raw data and cleaning decisions are not modified. No acoustic predictors, interactions, random slopes, song random effects, or participant metadata predictors are added here.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif not (PROJECT_ROOT / "statistical-baseline").exists():
    PROJECT_ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "statistical-baseline").exists())

sys.path.insert(0, str(PROJECT_ROOT / "statistical-baseline" / "src"))

from statistical_baseline.real_stimulus_model import (
    FINAL_FORMULA,
    INTERCEPT_FORMULA,
    OUTPUT_DIR,
    RATINGS_PATH,
    run_real_stimulus_model,
)

RATINGS_PATH, OUTPUT_DIR, INTERCEPT_FORMULA, FINAL_FORMULA

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`


WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


(WindowsPath('C:/Users/oscar/Documents/7. QMUL UNIVERSITY/1. Master Program/3. MSc Project/intent2control-dissertation/statistical-baseline/data/real/real_ratings_clean.csv'),
 WindowsPath('C:/Users/oscar/Documents/7. QMUL UNIVERSITY/1. Master Program/3. MSc Project/intent2control-dissertation/statistical-baseline/outputs/real_stimulus_model'),
 'rating ~ 1 + (1 | participant_id) + (1 | stimulus_id)',
 'rating ~ episode + group + (1 | participant_id) + (1 | stimulus_id)')

## Execute Locked Empirical Model Pipeline

Sampling settings are 4 chains, 1000 tuning draws, 1000 posterior draws, `target_accept=0.95`, deterministic seed, and nutpie where available. If nutpie is unavailable, the code records the PyMC fallback reason rather than changing the scientific model.

In [2]:
outputs = run_real_stimulus_model()
outputs.keys()

C:\Users\oscar\miniconda3\lib\site-packages\pymc\sampling\mcmc.py:328: UserWarning: `idata_kwargs` are currently ignored by the nutpie sampler
  warnings.warn(


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.25,15
,2000,0,0.23,15
,2000,0,0.23,31
,2000,0,0.23,15


C:\Users\oscar\miniconda3\lib\site-packages\pymc\sampling\mcmc.py:328: UserWarning: `idata_kwargs` are currently ignored by the nutpie sampler
  warnings.warn(


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.23,15
,2000,0,0.22,15
,2000,0,0.21,15
,2000,0,0.21,15


dict_keys(['summary', 'validation', 'descriptive', 'intercept_idata', 'final_idata', 'runtimes', 'diagnostics', 'variance', 'icc', 'fixed', 'episode_means', 'episode_contrasts', 'ppc', 'expected', 'winners', 'winner_validation', 'observed_preferences', 'findings_text'])

## Modelling Dataset Validation

In [3]:
outputs["validation"]

,check,passed,actual_value
0,final_analysable_n_30,True,30
1,group_01_n_16,True,16
2,group_02_n_14,True,14
3,rating_observations_900,True,900
4,stimuli_20,True,20
5,songs_4,True,4
6,episodes_3,True,3
7,ratings_per_participant_30,True,"{""count"": 30.0, ""mean"": 30.0, ""std"": 0.0, ""min..."
8,no_missing_outcome,True,0
9,no_invalid_ratings,True,0


## Real Descriptive Summaries

In [4]:
outputs["descriptive"].head(40)

,summary_type,level,n,mean,median,sd,min,max
0,overall,all,900,53.217778,55.00,32.134093,0.000000,100.000000
1,episode,EDR-1,300,54.580000,58.00,31.261925,0.000000,100.000000
2,episode,EDR-2,300,56.763333,62.50,32.504808,0.000000,100.000000
3,episode,FM-1,300,48.310000,50.00,32.127785,0.000000,100.000000
4,group,group_01,480,49.939583,50.00,30.648974,0.000000,100.000000
5,group,group_02,420,56.964286,68.00,33.395742,0.000000,100.000000
6,stimulus,id_like_to_know_pxl_s1,48,49.270833,52.50,33.639984,0.000000,100.000000
7,stimulus,id_like_to_know_pxl_s2,48,60.916667,60.50,26.298478,0.000000,100.000000
8,stimulus,id_like_to_know_pxl_s3,48,55.541667,59.00,35.077397,0.000000,100.000000
9,stimulus,id_like_to_know_pxl_s5,48,43.500000,46.50,26.075178,0.000000,100.000000


## Empirical ICCs and Variance Components

Participant ICC is the proportion of residual rating variance associated with stable between-listener differences. Stimulus ICC is the proportion associated with differences among the 20 mix stimuli.

In [5]:
outputs["variance"], outputs["icc"]

(                     term    mean     sd   hdi_3  hdi_97   mcse  mcse_sd  \
 0  1|participant_id_sigma  11.551  1.908   8.198  15.208  0.060    0.035   
 1     1|stimulus_id_sigma  13.830  2.708   9.029  18.839  0.110    0.074   
 2                   sigma  27.661  0.678  26.406  28.918  0.009    0.012   
 
    ess_bulk  ess_tail  r_hat  
 0    1001.0    1788.0    1.0  
 1     629.0     968.0    1.0  
 2    5473.0    2824.0    1.0  ,
                    term     mean      sd    hdi_3   hdi_97   mcse  mcse_sd  \
 0  participant_variance  137.074  46.170   64.248  227.074  1.431    1.037   
 1     stimulus_variance  198.613  81.875   79.741  351.900  3.296    3.149   
 2     residual_variance  765.579  37.611  697.277  836.241  0.516    0.682   
 3       participant_ICC    0.124   0.037    0.061    0.195  0.001    0.001   
 4          stimulus_ICC    0.177   0.057    0.084    0.287  0.002    0.002   
 5        residual_share    0.699   0.056    0.590    0.798  0.002    0.001   
 
    es

## Fixed Effects, Episode Means, and Episode Contrasts

In [6]:
outputs["fixed"], outputs["episode_means"], outputs["episode_contrasts"]

(              term    mean     sd   hdi_3  hdi_97   mcse  mcse_sd  ess_bulk  \
 0        Intercept  51.388  5.720  40.503  61.826  0.267    0.189     463.0   
 1   episode[EDR-2]   2.147  2.269  -2.119   6.462  0.043    0.030    2769.0   
 2    episode[FM-1]  -6.292  2.200 -10.390  -2.144  0.041    0.029    2874.0   
 3  group[group_02]   6.868  7.735  -6.425  22.576  0.349    0.198     492.0   
 
    ess_tail  r_hat  
 0     557.0   1.01  
 1    2797.0   1.00  
 2    2828.0   1.00  
 3     921.0   1.01  ,
   summary_type  level       mean     median        sd      hdi_3     hdi_97  \
 0      episode  EDR-1  54.593521  54.510070  4.310200  46.533787  62.849524   
 1      episode  EDR-2  56.740610  56.709797  4.330556  48.434418  64.839720   
 2      episode   FM-1  48.301966  48.267481  4.329634  40.013428  56.355321   
 
    probability_above_zero  probability_below_zero  
 0                     1.0                     0.0  
 1                     1.0                     0.0  
 2    

## Convergence Diagnostics

In [7]:
outputs["diagnostics"]

,model_name,divergences,max_rhat,min_bulk_ess,min_tail_ess,max_tree_depth,tree_depth_warnings,energy_warnings,runtime_seconds,sampler
0,intercept_only,0,1.01,463.0,695.0,NaN,0,0,24.531825,bambi_pymc_nutpie
1,final_stimulus,0,1.01,463.0,557.0,NaN,0,0,20.077196,bambi_pymc_nutpie


## Posterior Predictive Checks

Posterior predictive samples are not clipped; outside-range rates diagnose whether the Gaussian likelihood is materially strained by the bounded 0-100 scale.

In [8]:
outputs["ppc"]

,metric,value
0,observed_mean,53.217778
1,observed_sd,32.134093
2,posterior_predictive_mean_mean,53.218572
3,posterior_predictive_sd_mean,32.285727
4,posterior_predictive_below_0,0.050232
5,posterior_predictive_above_100,0.073403
6,posterior_predictive_outside_0_100,0.123635
7,observed_episode_mean_EDR-1,54.580000
8,ppc_episode_mean_EDR-1,54.622556
9,observed_episode_mean_EDR-2,56.763333


## Posterior Expected Ratings and Highest-Rated Mix Probabilities

Posterior highest-rated-mix probabilities compare the five valid stimuli within each `song_id x episode` combination for every posterior expected-rating draw. Exact draw-level ties receive fractional credit.

Because the model is additive and contains no Episode x Stimulus interaction, stimulus differences are assumed constant across episodes on the model scale. The model can produce episode-specific expected ratings through overall episode shifts, but it should not be interpreted as evidence for context-dependent stimulus reversals.

In [9]:
outputs["expected"].head(), outputs["winners"].head(20), outputs["winner_validation"]

(      group episode          song_id             stimulus_id  \
 0  group_01   EDR-1  id_like_to_know  id_like_to_know_pxl_s1   
 1  group_01   EDR-1  id_like_to_know  id_like_to_know_pxl_s2   
 2  group_01   EDR-1  id_like_to_know  id_like_to_know_pxl_s3   
 3  group_01   EDR-1  id_like_to_know  id_like_to_know_pxl_s5   
 4  group_01   EDR-1  id_like_to_know  id_like_to_know_pxl_s7   
 
              mix_id       mean     median        sd      hdi_3     hdi_97  \
 0  mix_9ffcd672af17  50.781493  50.692609  5.024594  41.260777  59.898847   
 1  mix_6f8b655d7999  61.416668  61.411965  4.872764  52.402837  70.804840   
 2  mix_65f41f41d3ab  56.609088  56.627627  4.867825  47.438054  65.619209   
 3  mix_82d60d7c7fc5  45.454261  45.352729  4.894738  36.227383  54.648943   
 4  mix_b393d3919c78  52.183252  52.212213  4.820300  42.890505  61.212225   
 
    probability_above_zero  probability_below_zero  
 0                     1.0                     0.0  
 1                     1.0      

## Observed Human Preference Summary

This table is an in-sample descriptive fit aid only. It is not out-of-sample predictive performance.

In [10]:
outputs["observed_preferences"].head(40)

,song_id,group,episode,stimulus_id,mix_id,observed_winner_credit,observed_winner_trials,tied_winner_rows,song_episode_trials,observed_winner_share_fractional_ties,comparison_scope
0,id_like_to_know,group_01,EDR-1,id_like_to_know_pxl_s1,mix_9ffcd672af17,6.000000,6,0,16,0.375000,in-sample descriptive fit only
2,id_like_to_know,group_01,EDR-1,id_like_to_know_pxl_s3,mix_65f41f41d3ab,5.000000,5,0,16,0.312500,in-sample descriptive fit only
3,id_like_to_know,group_01,EDR-1,id_like_to_know_pxl_s7,mix_b393d3919c78,3.000000,3,0,16,0.187500,in-sample descriptive fit only
1,id_like_to_know,group_01,EDR-1,id_like_to_know_pxl_s2,mix_6f8b655d7999,2.000000,2,0,16,0.125000,in-sample descriptive fit only
5,id_like_to_know,group_01,EDR-2,id_like_to_know_pxl_s2,mix_6f8b655d7999,5.700000,7,2,16,0.356250,in-sample descriptive fit only
6,id_like_to_know,group_01,EDR-2,id_like_to_know_pxl_s3,mix_65f41f41d3ab,4.700000,6,2,16,0.293750,in-sample descriptive fit only
4,id_like_to_know,group_01,EDR-2,id_like_to_know_pxl_s1,mix_9ffcd672af17,2.700000,4,2,16,0.168750,in-sample descriptive fit only
8,id_like_to_know,group_01,EDR-2,id_like_to_know_pxl_s7,mix_b393d3919c78,1.700000,3,2,16,0.106250,in-sample descriptive fit only
7,id_like_to_know,group_01,EDR-2,id_like_to_know_pxl_s5,mix_82d60d7c7fc5,1.200000,2,1,16,0.075000,in-sample descriptive fit only
11,id_like_to_know,group_01,FM-1,id_like_to_know_pxl_s3,mix_65f41f41d3ab,6.200000,7,1,16,0.387500,in-sample descriptive fit only


## RQ2 Relevance

RQ2 asks: "To what extent do mix preferences vary between listeners and within the same listener across listening contexts?"

The current stimulus model informs RQ2 through the empirical participant ICC, participant random-effect variance, episode fixed effects, stimulus variance, and descriptive preference distributions. However, because the primary model contains no participant-specific episode slopes and no Episode x Stimulus interactions, it does not fully parameterise every form of within-listener context-specific heterogeneity.

## Sample-Size Context

The planned preferred analysable sample was N=50. The achieved analysable sample is N=30. This does not invalidate the study, but it means posterior estimates may be less precise than anticipated in the design simulations. The sample-size design analysis is not rerun retroactively to make N=30 appear planned.

## Empirical Stimulus-Model Findings

In [11]:
print(outputs["findings_text"])
print("\nExport directory:", OUTPUT_DIR)

# Empirical Stimulus-Model Findings

The achieved analysable sample was N=30, with 16 participants in group_01 and 14 in group_02.
The empirical participant ICC posterior mean was 0.124 with 94% HDI [0.061, 0.195], representing the proportion of residual rating variance associated with stable between-listener differences.
The empirical stimulus ICC posterior mean was 0.177 with 94% HDI [0.084, 0.287], representing the proportion associated with differences among the 20 mix stimuli. The residual share posterior mean was 0.699.
Episode fixed effects are additive context shifts in native rating-point units, averaged across group and stimulus variability; they should be interpreted by magnitude, direction, and uncertainty rather than binary significance language.
The group coefficient is structurally linked to song allocation, so it represents a systematic difference between the two assigned song sets / study groups rather than an independent causal effect of group membership.
The final mo